# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"Dataset name: {metadata['name']}")
print(f"Dataset description: {metadata['description']}")
print(f"Dataset ID (@id): {metadata['@id']}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

`mlcroissant` provides structured access to each RecordSet defined in the Croissant schema. Each entity is referenced by its `@id`.

In [ ]:
# Inspect record sets, fields, and columns by their @id
record_sets = dataset.metadata.record_sets
print(f"Available RecordSets ({len(record_sets)}):")

# Show all RecordSet @id and their fields
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '')}")
    fields = rs.get('fields', [])
    print(f"  Fields:")
    for field in fields:
        print(f"    - Field @id: {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
    columns = rs.get('columns', [])
    if columns:
        print(f"  Columns:")
        for column in columns:
            print(f"    - Column @id: {column['@id']} (name: {column.get('name', '')}, dataType: {column.get('dataType', '')})")

## 3. Data Extraction
Load data from record sets into DataFrames. All entity references use the `@id` field.

Below, we extract all record sets found in the dataset and load their data.

In [ ]:
# Extract data from all available RecordSets by their @id
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Failed to load RecordSet {record_set_id}: {e}")

# If at least one record set is loaded, show the columns and preview
first_loaded_rs = next(iter(dataframes.keys()), None)
if first_loaded_rs:
    print(f"\nColumns in RecordSet {first_loaded_rs}:")
    print(dataframes[first_loaded_rs].columns.tolist())
    display(dataframes[first_loaded_rs].head())
else:
    print('No RecordSets loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing numeric fields, and grouping by key attributes. All variables referenced by their `@id`.

Below, we identify a numeric field for analysis and demonstrate filtering, normalization, and grouping. Replace the example IDs with IDs found from the RecordSet overview above if needed.

In [ ]:
# Pick one RecordSet for EDA (use the first loaded RecordSet)
record_set_id = first_loaded_rs
df = dataframes[record_set_id]

# Find numeric fields (based on Croissant schema 'dataType')
rs_obj = next((rs for rs in dataset.metadata.record_sets if rs['@id'] == record_set_id), None)
numeric_field_id = None
for field in rs_obj.get('fields', []):
    dt = field.get('dataType', '')
    if dt in ['schema:Integer', 'schema:Float', 'schema:Number']:
        numeric_field_id = field['@id']
        break

# If no numeric field found, skip EDA
if numeric_field_id is not None and numeric_field_id in df.columns:
    print(f"Using numeric field {numeric_field_id}")
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Find a field to group by (categorical/string)
    group_field_id = None
    for field in rs_obj.get('fields', []):
        dt = field.get('dataType', '')
        if dt in ['schema:Text', 'schema:Boolean'] and field['@id'] in df.columns:
            group_field_id = field['@id']
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found in RecordSet for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Entities are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution if suitable
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot with group field if available
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No suitable numeric field found for visualization.')

## 6. Conclusion
This notebook demonstrated loading, exploring, and analyzing the FAIR^2 clinical dataset using the `mlcroissant` library.
- Each data entity was referenced consistently by its `@id` field.
- The analysis included metadata exploration, record set identification, EDA on numeric variables, and visualizations.
- Further domain-driven exploration can be performed based on key clinical features from the dataset, such as biomarker prevalence or group comparisons.